# 02 — Build the Master Panel Dataset

**Dissertation:** *Agentic AI for Sovereign Risk Assessment under Climate-Related Fiscal Stress*

This notebook assembles the master panel dataset that the RL environment will consume.  
The merge produces **one row per (country, year)** containing all state variables,  
reward-function inputs, climate shock indicators, and classification labels.

**RL state space reminder:** output growth, debt-to-GDP, primary balance, interest rate,  
climate shock indicator, adaptation capital, risk premium.

**Merge strategy:** WEO is the base (widest country-year coverage of any single source).  
Every other dataset is left-joined onto it — no WEO row is ever discarded.

**Sources:**

| Source | File | Type |
|--------|------|------|
| IMF WEO | `data/processed/weo_imf.csv` | Panel (iso, year) |
| IMF Central Govt Debt | `data/processed/imf_central_govt_debt_panel.csv` | Panel (iso3, year) |
| World Bank WDI | `data/raw/world_bank_data.csv` | Panel (country=ISO3, year) |
| EM-DAT | `data/raw/emdat_disaster_data.xlsx` | Event-level → aggregate |
| ND-GAIN | 6 CSVs in `data/raw/notre_dame-gain_countryindex_2026/` | Wide → panel |
| WB Income Class | `data/raw/wbg_income_class_2025_10_07.xlsx` | Cross-sectional |
| INFORM Risk | `data/raw/European_commission-INFORM_Risk_Mid_2025_v071.xlsx` | Cross-sectional |

**Output:** `data/processed/master_panel.csv`

---
## Setup — imports, paths, logging

All paths are resolved relative to the project root so this notebook runs  
correctly regardless of where Jupyter is launched from.

In [ ]:
import sys
import logging
import warnings
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")


def find_project_root(start: Path = Path().resolve()) -> Path:
    for directory in [start, *start.parents]:
        if (directory / "config").is_dir():
            return directory
    raise FileNotFoundError(
        "Cannot find project root — expected a config/ folder somewhere above."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import LOG_FORMAT, LOG_DATE_FORMAT

logging.basicConfig(
    level=logging.INFO,
    format=LOG_FORMAT,
    datefmt=LOG_DATE_FORMAT,
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("master_panel")

RAW     = PROJECT_ROOT / "data" / "raw"
PROC    = PROJECT_ROOT / "data" / "processed"
NDGAIN  = RAW / "notre_dame-gain_countryindex_2026" / "resources"

PROC.mkdir(parents=True, exist_ok=True)

log.info("Project root : %s", PROJECT_ROOT)
log.info("Raw data     : %s", RAW)
log.info("Processed    : %s", PROC)

### Helper: ISO standardisation

Every dataset uses ISO3 as the merge key. Before any join we normalise every ISO column  
to uppercase, stripped of whitespace, and verify it is exactly 3 characters.  
Codes that fail the length check are printed so they can be investigated.

In [ ]:
def standardise_iso(df: pd.DataFrame, col: str = "iso3") -> pd.DataFrame:
    """Uppercase, strip whitespace, then flag any codes that aren't 3 chars."""
    df = df.copy()
    df[col] = df[col].astype(str).str.strip().str.upper()
    df[col] = df[col].replace("NAN", pd.NA)
    bad = df.loc[df[col].notna() & (df[col].str.len() != 3), col].unique()
    if len(bad):
        log.warning("Non-3-char ISO codes in column '%s': %s", col, bad.tolist())
    return df


def match_rate(base: pd.DataFrame, joined: pd.DataFrame,
               sentinel_col: str) -> str:
    """Return a formatted match-rate string."""
    n_base    = len(base)
    n_matched = joined[sentinel_col].notna().sum()
    pct       = n_matched / n_base * 100
    return f"{n_matched:,} / {n_base:,} rows matched ({pct:.1f}%)"


log.info("Helper functions ready.")

---
## Step 1 — Load WEO processed data and select relevant columns

The WEO panel (`weo_imf.csv`) was produced by `src/notebooks/weo_data_extraction.ipynb`.  
It contains 28 subject descriptor columns plus `iso` (ISO3 country code) and `year`.

**Why column selection matters here:**  
We keep only the 15 variables that feed directly into the RL state space, reward function,  
or feature engineering pipeline. Retaining irrelevant columns (PPP-based GDP, employment,  
end-of-period inflation, etc.) would bloat the master panel and create confusion later.

The WEO `iso` column is renamed to `iso3` to match the standard merge key used across  
all datasets in this project.

In [ ]:
WEO_FILE = PROC / "weo_imf.csv"

try:
    weo_raw = pd.read_csv(WEO_FILE)
    log.info("WEO loaded: %d rows × %d columns", *weo_raw.shape)
    print("dtypes:")
    print(weo_raw.dtypes.to_string())
    print("\nFirst 3 rows:")
    print(weo_raw.head(3).to_string(index=False))
except FileNotFoundError as e:
    log.error("WEO file not found: %s", e)
    raise

In [ ]:
# Columns to keep and their target names.
# The dict key is the exact column name in weo_imf.csv.
WEO_KEEP = {
    "iso":                                              "iso3",
    "year":                                             "year",
    "Gross domestic product, constant prices":          "weo_gdp_constant",
    "Gross domestic product, current prices":           "weo_gdp_current",
    "Gross domestic product per capita, current prices":"weo_gdp_per_capita",
    "Gross domestic product, deflator":                 "weo_gdp_deflator",
    "General government gross debt":                    "weo_govt_gross_debt",
    "General government primary net lending/borrowing": "weo_primary_balance",
    "General government revenue":                       "weo_govt_revenue",
    "General government total expenditure":             "weo_govt_expenditure",
    "General government net lending/borrowing":         "weo_fiscal_balance",
    "General government structural balance":            "weo_structural_balance",
    "Output gap in percent of potential GDP":           "weo_output_gap",
    "Current account balance":                         "weo_current_account",
    "Inflation, average consumer prices":              "weo_inflation_cpi",
    "Population":                                      "weo_population",
    "Unemployment rate":                               "weo_unemployment",
}

# Verify all expected columns are present before selecting
missing_weo = [c for c in WEO_KEEP if c not in weo_raw.columns]
if missing_weo:
    log.error("MISSING WEO columns: %s", missing_weo)
    log.error("Available: %s", weo_raw.columns.tolist())
else:
    log.info("All %d WEO columns found.", len(WEO_KEEP))

dropped_weo = [c for c in weo_raw.columns if c not in WEO_KEEP]
log.info("Dropping %d WEO columns: %s", len(dropped_weo), dropped_weo)

weo = (
    weo_raw[list(WEO_KEEP.keys())]
    .rename(columns=WEO_KEEP)
    .copy()
)
weo = standardise_iso(weo, "iso3")

print(f"\nWEO selected: {weo.shape[0]:,} rows × {weo.shape[1]} columns")
print("Columns:", weo.columns.tolist())
print(f"Countries: {weo['iso3'].nunique()}  |  Years: {weo['year'].min()}–{weo['year'].max()}")

---
## Step 2 — Load IMF Central Government Debt processed data

This panel was produced in `notebooks/01_reshape_imf_debt.ipynb`.  
It provides **central government** debt specifically, which is more comparable  
across countries than general government debt (the WEO metric). We carry both  
and let the consolidation step in Step 8 resolve priority.

We rename `debt_to_gdp` → `imf_cg_debt_to_gdp` to make the source explicit  
and prevent column-name collisions during the merge.

In [ ]:
IMF_DEBT_FILE = PROC / "imf_central_govt_debt_panel.csv"

try:
    imf_debt_raw = pd.read_csv(IMF_DEBT_FILE)
    log.info("IMF debt loaded: %d rows × %d columns", *imf_debt_raw.shape)
    print("First 3 rows:")
    print(imf_debt_raw.head(3).to_string(index=False))
except FileNotFoundError as e:
    log.error("IMF debt file not found: %s", e)
    raise

# Keep only the three merge-relevant columns
imf_debt = (
    imf_debt_raw[["iso3", "year", "debt_to_gdp"]]
    .rename(columns={"debt_to_gdp": "imf_cg_debt_to_gdp"})
    .copy()
)

# Standardise
imf_debt = standardise_iso(imf_debt, "iso3")
imf_debt["year"] = pd.to_numeric(imf_debt["year"], errors="coerce").astype("Int64")
imf_debt["imf_cg_debt_to_gdp"] = pd.to_numeric(imf_debt["imf_cg_debt_to_gdp"], errors="coerce")

print(f"\nIMF debt selected: {imf_debt.shape[0]:,} rows × {imf_debt.shape[1]} columns")
print(f"Year range   : {int(imf_debt['year'].min())} – {int(imf_debt['year'].max())}")
print(f"Unique countries: {imf_debt['iso3'].nunique()}")

---
## Step 3 — Load World Bank WDI data and select relevant columns

The World Bank panel (`world_bank_data.csv`) was downloaded via `wbgapi` in  
`src/notebooks/wbg_data.ipynb`. It covers 1990–2024 for ~266 economies.

**Important:** The `country` column in this file already contains ISO 3-letter codes  
(e.g. `ABW`, `AFG`) — not country names. This is because `wbgapi` returns economy codes  
directly. We simply rename it to `iso3`; no `pycountry` lookup is needed here.

We drop `gdp_per_capita_usd` (already in WEO as `weo_gdp_per_capita`), `population`  
(already in WEO), `exports_gdp`, `imports_gdp` (the current account captures the net  
external position which is what the state space needs), and `foreign_reserves_usd`  
(not in the state space).

In [ ]:
WB_FILE = RAW / "world_bank_data.csv"

try:
    wb_raw = pd.read_csv(WB_FILE)
    log.info("WB WDI loaded: %d rows × %d columns", *wb_raw.shape)
    print("dtypes:")
    print(wb_raw.dtypes.to_string())
    print("\nFirst 3 rows:")
    print(wb_raw.head(3).to_string(index=False))
except FileNotFoundError as e:
    log.error("WB file not found: %s", e)
    raise

In [ ]:
WB_KEEP = {
    "country":             "iso3",               # already ISO3 codes from wbgapi
    "year":                "year",
    "gdp_current_usd":     "wdi_gdp_current_usd",
    "gdp_growth":          "wdi_gdp_growth",
    "debt_to_gdp":         "wdi_debt_to_gdp",
    "tax_revenue_gdp":     "wdi_tax_revenue_gdp",
    "gov_expenditure_gdp": "wdi_gov_expenditure_gdp",
    "inflation_cpi":       "wdi_inflation_cpi",
    "inflation_deflator":  "wdi_inflation_deflator",
    "current_account_gdp": "wdi_current_account_gdp",
}

missing_wb = [c for c in WB_KEEP if c not in wb_raw.columns]
if missing_wb:
    log.error("MISSING WB columns: %s", missing_wb)
else:
    log.info("All %d WB columns found.", len(WB_KEEP))

dropped_wb = [c for c in wb_raw.columns if c not in WB_KEEP]
log.info("Dropping %d WB columns: %s", len(dropped_wb), dropped_wb)

wb = (
    wb_raw[list(WB_KEEP.keys())]
    .rename(columns=WB_KEEP)
    .copy()
)
wb = standardise_iso(wb, "iso3")
wb["year"] = pd.to_numeric(wb["year"], errors="coerce").astype("Int64")

print(f"\nWB WDI selected: {wb.shape[0]:,} rows × {wb.shape[1]} columns")
print("Columns:", wb.columns.tolist())
print(f"Countries: {wb['iso3'].nunique()}  |  Years: {int(wb['year'].min())}–{int(wb['year'].max())}")

---
## Step 4 — Aggregate EM-DAT to country-year panel

EM-DAT is an **event-level** dataset: each row is a single disaster event.  
For the RL environment we need a **country-year** summary — how many events,  
how many deaths, how many affected, and the economic cost.

**Pre-inspection findings (confirmed before writing this notebook):**
- The file has already been filtered to climate-related disasters at source.  
  `Disaster Subgroup` contains only: Hydrological, Climatological, Meteorological.
- The `ISO` column already contains ISO 3-letter codes (all length-3, verified).
- Years range from 2000 to 2026.

Country-years with zero climate disasters will simply be absent from this panel.  
After the left-join to WEO they become NaN — which is correct; during feature  
engineering we will fill `climate_disaster_count` NaN → 0.

In [ ]:
EMDAT_FILE = RAW / "emdat_disaster_data.xlsx"

EMDAT_KEEP = [
    "ISO",
    "Start Year",
    "Disaster Subgroup",
    "Total Deaths",
    "Total Affected",
    "Total Damage ('000 US$)",
    "Total Damage, Adjusted ('000 US$)",
]

try:
    emdat_raw = pd.read_excel(
        EMDAT_FILE,
        sheet_name="EM-DAT Data",
        dtype=str,
        engine="openpyxl",
    )
    log.info("EM-DAT loaded: %d rows × %d columns", *emdat_raw.shape)
    print("First 3 rows (key columns):")
    print(emdat_raw[EMDAT_KEEP].head(3).to_string(index=False))
except Exception as e:
    log.error("EM-DAT load failed: %s", e)
    raise

In [ ]:
# ── Verify Disaster Subgroup ──────────────────────────────────────────────────
observed_subgroups = emdat_raw["Disaster Subgroup"].unique().tolist()
print("Unique Disaster Subgroup values:", observed_subgroups)

EXPECTED_SUBGROUPS = {"Climatological", "Meteorological", "Hydrological"}
unexpected = set(observed_subgroups) - EXPECTED_SUBGROUPS - {None, "nan"}
if unexpected:
    log.warning("Unexpected subgroups found: %s — filtering them out.", unexpected)
    emdat_raw = emdat_raw[emdat_raw["Disaster Subgroup"].isin(EXPECTED_SUBGROUPS)].copy()
    log.info("After filter: %d rows", len(emdat_raw))
else:
    log.info("Subgroup check passed — all events are climate-related.")

In [ ]:
# ── Drop unneeded columns immediately ────────────────────────────────────────
emdat = emdat_raw[EMDAT_KEEP].copy()
log.info("After column selection: %d rows × %d columns", *emdat.shape)

# ── Convert numeric columns — log coercions ───────────────────────────────────
NUMERIC_COLS = [
    "Total Deaths",
    "Total Affected",
    "Total Damage ('000 US$)",
    "Total Damage, Adjusted ('000 US$)",
]

print("\nCoercions to NaN per column:")
for col in NUMERIC_COLS:
    before_na = emdat[col].isna().sum()
    emdat[col] = pd.to_numeric(
        emdat[col].astype(str).str.strip().str.replace(",", "", regex=False),
        errors="coerce",
    )
    after_na   = emdat[col].isna().sum()
    print(f"  {col:<42}: {after_na - before_na:>5} values coerced")

# ── ISO column: already ISO3, just rename and standardise ─────────────────────
emdat = emdat.rename(columns={"ISO": "iso3"})
emdat = standardise_iso(emdat, "iso3")

# ── Year column ──────────────────────────────────────────────────────────────
emdat = emdat.rename(columns={"Start Year": "year"})
emdat["year"] = pd.to_numeric(emdat["year"], errors="coerce").astype("Int64")

print(f"\nBefore aggregation: {len(emdat):,} event rows")

In [ ]:
# ── Aggregate to country-year ─────────────────────────────────────────────────
# climate_disaster_count = number of events (rows) per (iso3, year)
# All other columns are summed — NaN in a sum is treated as 0 by default
# (min_count=1 ensures that a group of all-NaN values stays NaN rather than 0)

emdat_panel = emdat.groupby(["iso3", "year"], as_index=False).agg(
    climate_disaster_count=("Disaster Subgroup", "count"),
    climate_deaths=("Total Deaths", "sum"),
    climate_affected=("Total Affected", "sum"),
    climate_damage_000usd=("Total Damage ('000 US$)", "sum"),
    climate_damage_adj_000usd=("Total Damage, Adjusted ('000 US$)", "sum"),
)

emdat_panel = emdat_panel.sort_values(["iso3", "year"]).reset_index(drop=True)

print(f"EM-DAT panel: {emdat_panel.shape[0]:,} rows × {emdat_panel.shape[1]} columns")
print(f"Year range  : {int(emdat_panel['year'].min())} – {int(emdat_panel['year'].max())}")
print(f"Countries   : {emdat_panel['iso3'].nunique()}")
print("\nFirst 5 rows:")
print(emdat_panel.head(5).to_string(index=False))

---
## Step 5 — Reshape and merge ND-GAIN aggregate files

Notre Dame Global Adaptation Initiative (ND-GAIN) publishes composite indices  
measuring a country's vulnerability to climate change and its readiness to adapt.  
These map directly to two state variables:

- **`ndgain_readiness`** → proxy for **adaptation capital** in the RL state space  
- **`ndgain_vulnerability`** → proxy for **climate shock indicator** baseline level

All 6 ND-GAIN files share the same wide format: `ISO3`, `Name`, then year columns  
`1995`, `1996`, …, `2023`. We write one reusable reshape function and apply it to  
each file, then outer-join the 6 panels together (since coverage is identical across  
all 6 files, any join type gives the same result — we use outer for safety).

In [ ]:
def load_ndgain(file_path: Path, var_name: str) -> pd.DataFrame:
    """
    Load one ND-GAIN wide CSV and reshape to panel format.

    Parameters
    ----------
    file_path : Path to the ND-GAIN CSV file.
    var_name  : Name to give the value column in the output panel.

    Returns
    -------
    DataFrame with columns [iso3, year, var_name].
    """
    if not file_path.exists():
        log.error("ND-GAIN file not found: %s", file_path)
        return pd.DataFrame(columns=["iso3", "year", var_name])

    df = pd.read_csv(file_path, dtype=str)

    # Identify year columns: 4-digit integers between 1900–2100
    def is_year(c):
        try:
            yr = int(str(c).strip())
            return 1900 <= yr <= 2100
        except (ValueError, TypeError):
            return False

    year_cols = [c for c in df.columns if is_year(c)]
    keep_cols = ["ISO3", "Name"] + year_cols
    df = df[[c for c in keep_cols if c in df.columns]]

    # Melt to long
    long = df.melt(
        id_vars=["ISO3", "Name"],
        value_vars=year_cols,
        var_name="year",
        value_name=var_name,
    )

    long = long.rename(columns={"ISO3": "iso3"}).drop(columns=["Name"])
    long = standardise_iso(long, "iso3")
    long["year"]   = pd.to_numeric(long["year"],   errors="coerce").astype("Int64")
    long[var_name] = pd.to_numeric(long[var_name], errors="coerce")
    long = long.dropna(subset=["iso3", "year"]).sort_values(["iso3", "year"]).reset_index(drop=True)

    log.info("  %-28s : %d rows, %d countries, years %d–%d",
             var_name, len(long), long["iso3"].nunique(),
             int(long["year"].min()), int(long["year"].max()))
    return long


log.info("load_ndgain() ready.")

In [ ]:
NDGAIN_FILES = {
    "ndgain_score":         NDGAIN / "gain"          / "gain.csv",
    "ndgain_readiness":     NDGAIN / "readiness"     / "readiness.csv",
    "ndgain_vulnerability": NDGAIN / "vulnerability" / "vulnerability.csv",
    "ndgain_exposure":      NDGAIN / "vulnerability" / "exposure.csv",
    "ndgain_sensitivity":   NDGAIN / "vulnerability" / "sensitivity.csv",
    "ndgain_capacity":      NDGAIN / "vulnerability" / "capacity.csv",
}

log.info("Loading %d ND-GAIN files...", len(NDGAIN_FILES))
ndgain_panels = {}
for var_name, path in NDGAIN_FILES.items():
    ndgain_panels[var_name] = load_ndgain(path, var_name)

In [ ]:
# Merge all 6 ND-GAIN panels together on (iso3, year)
ndgain_list = list(ndgain_panels.values())
ndgain = ndgain_list[0]

for df_next in ndgain_list[1:]:
    ndgain = ndgain.merge(df_next, on=["iso3", "year"], how="outer")

ndgain = ndgain.sort_values(["iso3", "year"]).reset_index(drop=True)

print(f"ND-GAIN combined: {ndgain.shape[0]:,} rows × {ndgain.shape[1]} columns")
print(f"Year range: {int(ndgain['year'].min())} – {int(ndgain['year'].max())}")
print(f"Countries : {ndgain['iso3'].nunique()}")
print("Columns   :", ndgain.columns.tolist())

---
## Step 6 — Load classification reference tables

These two cross-sectional tables attach country-level metadata to the panel.  
They join on `iso3` only (not `year`) — every row for a given country gets the  
same classification label regardless of year.

**World Bank income classification:** used to assign the economy type  
(`Advanced`, `Emerging Market`, `Developing`) that structures the RL's 3×3 matrix.

**INFORM Risk Index:** cross-validates the ND-GAIN climate vulnerability measure  
and provides hazard subcomponent scores for sensitivity analysis.

In [ ]:
WB_INCOME_FILE = RAW / "wbg_income_class_2025_10_07.xlsx"

try:
    wb_income_raw = pd.read_excel(
        WB_INCOME_FILE,
        sheet_name="List of economies",
        dtype=str,
        engine="openpyxl",
    )
    log.info("WB income loaded: %d rows × %d columns", *wb_income_raw.shape)
    print("Columns:", wb_income_raw.columns.tolist())
    print(wb_income_raw.head(3).to_string(index=False))
except Exception as e:
    log.error("WB income load failed: %s", e)
    raise

# Rename
wb_income = wb_income_raw.rename(columns={
    "Economy": "country_name",
    "Code":    "iso3",
}).copy()
wb_income = standardise_iso(wb_income, "iso3")

# Map income group → economy type
INCOME_TO_TYPE = {
    "High income":          "Advanced",
    "Upper middle income":  "Emerging Market",
    "Lower middle income":  "Developing",
    "Low income":           "Developing",
}
wb_income["economy_type"] = wb_income["Income group"].map(INCOME_TO_TYPE)

unmapped_income = wb_income[wb_income["economy_type"].isna() & wb_income["Income group"].notna()]
if not unmapped_income.empty:
    log.warning("Unmapped income groups: %s",
                unmapped_income["Income group"].unique().tolist())

print("\nCounts per economy_type:")
print(wb_income["economy_type"].value_counts(dropna=False).to_string())

In [ ]:
INFORM_FILE = RAW / "European_commission-INFORM_Risk_Mid_2025_v071.xlsx"

INFORM_KEEP = {
    "ISO3":                  "iso3",
    "INFORM RISK":           "inform_risk",
    "Natural":               "inform_natural_hazard",
    "Drought":               "inform_drought",
    "River Flood":           "inform_river_flood",
    "Tropical Cyclone":      "inform_tropical_cyclone",
    "Coastal flood":         "inform_coastal_flood",
    "VULNERABILITY":         "inform_vulnerability",
    "LACK OF COPING CAPACITY": "inform_lack_coping",
}

try:
    inform_raw = pd.read_excel(
        INFORM_FILE,
        sheet_name="INFORM Risk Mid 2025 (a-z)",
        dtype=str,
        engine="openpyxl",
    )
    log.info("INFORM loaded: %d rows × %d columns", *inform_raw.shape)
except Exception as e:
    log.error("INFORM load failed: %s", e)
    raise

# Verify expected columns exist
missing_inform = [c for c in INFORM_KEEP if c not in inform_raw.columns]
if missing_inform:
    log.error("MISSING INFORM columns: %s", missing_inform)
else:
    log.info("All INFORM columns found.")

dropped_inform = [c for c in inform_raw.columns if c not in INFORM_KEEP]
log.info("Dropping %d INFORM columns.", len(dropped_inform))

inform = (
    inform_raw[[c for c in INFORM_KEEP if c in inform_raw.columns]]
    .rename(columns=INFORM_KEEP)
    .copy()
)
inform = standardise_iso(inform, "iso3")

# Convert scores to numeric
for col in inform.columns:
    if col != "iso3":
        inform[col] = pd.to_numeric(inform[col], errors="coerce")

# Drop any row with no ISO3
inform = inform[inform["iso3"].notna()].reset_index(drop=True)

print(f"INFORM selected: {inform.shape[0]} rows × {inform.shape[1]} columns")
print("Columns:", inform.columns.tolist())
print("\nFirst 3 rows:")
print(inform.head(3).to_string(index=False))

---
## Step 7 — Execute the merge sequence

All merges use `how='left'` to preserve every WEO country-year row.  
Panel merges (on `iso3` + `year`) use `validate='1:1'` — if this fails it means  
a source dataset has duplicate (country, year) pairs, which must be investigated.  
Cross-sectional merges (on `iso3` only) use `validate='m:1'`.

After each join we print the new shape and match rate so you can immediately see  
how well each source covers the WEO base.

In [ ]:
# ── Base: WEO ─────────────────────────────────────────────────────────────────
master = weo.copy()
print(f"Base (WEO): {master.shape[0]:,} rows × {master.shape[1]} columns")

In [ ]:
# ── 7.1  Left-join IMF Central Government Debt ────────────────────────────────
try:
    master = master.merge(
        imf_debt,
        on=["iso3", "year"],
        how="left",
        validate="1:1",
    )
    log.info("IMF debt joined.")
except pd.errors.MergeError as e:
    log.error("Duplicate key error (IMF debt): %s", e)
    dupes = imf_debt[imf_debt.duplicated(["iso3", "year"], keep=False)]
    print(dupes.to_string())
    raise

print(f"After IMF debt : {master.shape[0]:,} rows × {master.shape[1]} columns")
print("Match rate     :", match_rate(weo, master, "imf_cg_debt_to_gdp"))

In [ ]:
# ── 7.2  Left-join World Bank WDI ─────────────────────────────────────────────
try:
    master = master.merge(
        wb,
        on=["iso3", "year"],
        how="left",
        validate="1:1",
    )
    log.info("WB WDI joined.")
except pd.errors.MergeError as e:
    log.error("Duplicate key error (WB WDI): %s", e)
    dupes = wb[wb.duplicated(["iso3", "year"], keep=False)]
    print(dupes.to_string())
    raise

print(f"After WB WDI   : {master.shape[0]:,} rows × {master.shape[1]} columns")
print("Match rate     :", match_rate(weo, master, "wdi_gdp_growth"))

In [ ]:
# ── 7.3  Left-join EM-DAT aggregated ─────────────────────────────────────────
try:
    master = master.merge(
        emdat_panel,
        on=["iso3", "year"],
        how="left",
        validate="1:1",
    )
    log.info("EM-DAT joined.")
except pd.errors.MergeError as e:
    log.error("Duplicate key error (EM-DAT): %s", e)
    dupes = emdat_panel[emdat_panel.duplicated(["iso3", "year"], keep=False)]
    print(dupes.to_string())
    raise

print(f"After EM-DAT   : {master.shape[0]:,} rows × {master.shape[1]} columns")
print("Match rate     :", match_rate(weo, master, "climate_disaster_count"))
print("(NaN = no climate disaster recorded in that country-year — expected.)")

In [ ]:
# ── 7.4  Left-join ND-GAIN ────────────────────────────────────────────────────
try:
    master = master.merge(
        ndgain,
        on=["iso3", "year"],
        how="left",
        validate="1:1",
    )
    log.info("ND-GAIN joined.")
except pd.errors.MergeError as e:
    log.error("Duplicate key error (ND-GAIN): %s", e)
    dupes = ndgain[ndgain.duplicated(["iso3", "year"], keep=False)]
    print(dupes.to_string())
    raise

print(f"After ND-GAIN  : {master.shape[0]:,} rows × {master.shape[1]} columns")
print("Match rate     :", match_rate(weo, master, "ndgain_score"))
print("(NaN for years outside 1995–2023 is expected.)")

In [ ]:
# ── 7.5  Left-join WB Income Classification (cross-sectional on iso3 only) ────
try:
    master = master.merge(
        wb_income[["iso3", "country_name", "Region", "Income group", "Lending category", "economy_type"]],
        on="iso3",
        how="left",
        validate="m:1",
    )
    log.info("Income classification joined.")
except pd.errors.MergeError as e:
    log.error("Duplicate key error (income): %s", e)
    dupes = wb_income[wb_income.duplicated("iso3", keep=False)]
    print(dupes.to_string())
    raise

n_no_class = master[master["economy_type"].isna()]["iso3"].nunique()
print(f"After income class : {master.shape[0]:,} rows × {master.shape[1]} columns")
print(f"Countries with no classification: {n_no_class}")
if n_no_class:
    unclassed = master[master["economy_type"].isna()]["iso3"].unique().tolist()
    print("  Unclassified ISO3 codes:", sorted(unclassed))

In [ ]:
# ── 7.6  Left-join INFORM Risk (cross-sectional on iso3 only) ─────────────────
try:
    master = master.merge(
        inform,
        on="iso3",
        how="left",
        validate="m:1",
    )
    log.info("INFORM joined.")
except pd.errors.MergeError as e:
    log.error("Duplicate key error (INFORM): %s", e)
    dupes = inform[inform.duplicated("iso3", keep=False)]
    print(dupes.to_string())
    raise

print(f"After INFORM       : {master.shape[0]:,} rows × {master.shape[1]} columns")
print(f"Match rate         : {master['inform_risk'].notna().sum():,} / {len(master):,} rows have INFORM data")

print("\n" + "=" * 60)
print(f"FINAL MASTER PANEL : {master.shape[0]:,} rows × {master.shape[1]} columns")
print("=" * 60)

---
## Step 8 — Create consolidated priority columns

We have multiple sources for the same underlying variable. Rather than pick one  
arbitrarily, we create a consolidated column using a **priority cascade**: use the  
best-coverage source first; fall back to the next when the primary is NaN.

This preserves maximum coverage while making the source hierarchy explicit.  
We print how many observations each source contributes so you can assess  
reliance on each source in your methodology chapter.

**Priority order rationale:**
- **Debt:** IMF Central Govt > WEO General Govt > WDI (IMF CG is most comparable  
  across countries; WEO general govt is broader but well-documented; WDI is sparse)
- **GDP growth:** computed from WEO constant prices (consistent methodology) > WDI direct
- **Inflation:** WEO CPI > WDI CPI (same concept, WEO covers more countries and years)
- **Current account:** WEO > WDI (same reasoning)

In [ ]:
# ── 8.1  Consolidated debt_to_gdp ────────────────────────────────────────────
# Priority: imf_cg_debt_to_gdp > weo_govt_gross_debt > wdi_debt_to_gdp

master["debt_to_gdp"] = np.nan

src1_mask = master["imf_cg_debt_to_gdp"].notna()
src2_mask = master["weo_govt_gross_debt"].notna() & ~src1_mask
src3_mask = master["wdi_debt_to_gdp"].notna() & ~src1_mask & ~src2_mask

master.loc[src1_mask, "debt_to_gdp"] = master.loc[src1_mask, "imf_cg_debt_to_gdp"]
master.loc[src2_mask, "debt_to_gdp"] = master.loc[src2_mask, "weo_govt_gross_debt"]
master.loc[src3_mask, "debt_to_gdp"] = master.loc[src3_mask, "wdi_debt_to_gdp"]

print("debt_to_gdp source contributions:")
print(f"  imf_cg_debt_to_gdp : {src1_mask.sum():>6,} rows")
print(f"  weo_govt_gross_debt : {src2_mask.sum():>6,} rows")
print(f"  wdi_debt_to_gdp     : {src3_mask.sum():>6,} rows")
print(f"  still NaN           : {master['debt_to_gdp'].isna().sum():>6,} rows")

In [ ]:
# ── 8.2  Consolidated gdp_growth ─────────────────────────────────────────────
# Compute year-on-year % change from WEO constant prices (within each country)
# then fall back to WDI direct measure.

master = master.sort_values(["iso3", "year"]).reset_index(drop=True)

master["weo_gdp_growth_computed"] = (
    master.groupby("iso3")["weo_gdp_constant"]
    .pct_change() * 100
)

master["gdp_growth"] = np.nan

src1_mask = master["weo_gdp_growth_computed"].notna()
src2_mask = master["wdi_gdp_growth"].notna() & ~src1_mask

master.loc[src1_mask, "gdp_growth"] = master.loc[src1_mask, "weo_gdp_growth_computed"]
master.loc[src2_mask, "gdp_growth"] = master.loc[src2_mask, "wdi_gdp_growth"]

print("gdp_growth source contributions:")
print(f"  weo_gdp_constant (computed pct_change) : {src1_mask.sum():>6,} rows")
print(f"  wdi_gdp_growth (direct)                : {src2_mask.sum():>6,} rows")
print(f"  still NaN                              : {master['gdp_growth'].isna().sum():>6,} rows")

In [ ]:
# ── 8.3  Consolidated inflation ───────────────────────────────────────────────
# Priority: weo_inflation_cpi > wdi_inflation_cpi

master["inflation"] = np.nan

src1_mask = master["weo_inflation_cpi"].notna()
src2_mask = master["wdi_inflation_cpi"].notna() & ~src1_mask

master.loc[src1_mask, "inflation"] = master.loc[src1_mask, "weo_inflation_cpi"]
master.loc[src2_mask, "inflation"] = master.loc[src2_mask, "wdi_inflation_cpi"]

print("inflation source contributions:")
print(f"  weo_inflation_cpi : {src1_mask.sum():>6,} rows")
print(f"  wdi_inflation_cpi : {src2_mask.sum():>6,} rows")
print(f"  still NaN         : {master['inflation'].isna().sum():>6,} rows")

In [ ]:
# ── 8.4  Consolidated current account (% GDP) ────────────────────────────────
# Priority: weo_current_account > wdi_current_account_gdp

master["current_account_gdp"] = np.nan

src1_mask = master["weo_current_account"].notna()
src2_mask = master["wdi_current_account_gdp"].notna() & ~src1_mask

master.loc[src1_mask, "current_account_gdp"] = master.loc[src1_mask, "weo_current_account"]
master.loc[src2_mask, "current_account_gdp"] = master.loc[src2_mask, "wdi_current_account_gdp"]

print("current_account_gdp source contributions:")
print(f"  weo_current_account     : {src1_mask.sum():>6,} rows")
print(f"  wdi_current_account_gdp : {src2_mask.sum():>6,} rows")
print(f"  still NaN               : {master['current_account_gdp'].isna().sum():>6,} rows")

# Drop the intermediate computed column — not needed in final panel
master = master.drop(columns=["weo_gdp_growth_computed"])

---
## Step 9 — Post-merge diagnostics

Before saving we run five checks:

1. **Column inventory** — full list of columns with total count.
2. **Coverage summary** — unique countries, year range, total rows.
3. **Missingness table** — sorted highest to lowest so the sparsest variables  
   are immediately visible.
4. **Economy type counts** — ensures the 3×3 classification is populated.
5. **Duplicate check** — any duplicate `(iso3, year)` pairs must be investigated  
   and resolved before the data can be used for training.

In [ ]:
# ── 9.1  Column inventory ─────────────────────────────────────────────────────
print(f"Total columns: {len(master.columns)}\n")
for i, col in enumerate(master.columns):
    print(f"  [{i+1:3d}] {col}")

In [ ]:
# ── 9.2  Coverage summary ─────────────────────────────────────────────────────
print("=" * 50)
print("COVERAGE SUMMARY")
print("=" * 50)
print(f"  Total rows        : {len(master):,}")
print(f"  Unique countries  : {master['iso3'].nunique()}")
print(f"  Year range        : {int(master['year'].min())} – {int(master['year'].max())}")

In [ ]:
# ── 9.3  Missingness rate per column (sorted highest first) ───────────────────
n_rows = len(master)
missing_pct = (
    master.isna().sum()
    .div(n_rows)
    .mul(100)
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["column", "missing_pct"]
missing_pct["missing_n"] = (missing_pct["missing_pct"] / 100 * n_rows).round().astype(int)

print(f"{'Column':<45} {'Missing N':>10} {'Missing %':>10}")
print("-" * 70)
for _, row in missing_pct.iterrows():
    flag = " *** VERY SPARSE" if row["missing_pct"] > 70 else (
           " *   SPARSE"      if row["missing_pct"] > 30 else "")
    print(f"{row['column']:<45} {row['missing_n']:>10,} {row['missing_pct']:>9.1f}%{flag}")

In [ ]:
# ── 9.4  Economy type distribution ───────────────────────────────────────────
print("Economy type counts (unique countries per type):")
eco_counts = (
    master.drop_duplicates("iso3")[["iso3", "economy_type"]]
    .groupby("economy_type", dropna=False)
    .size()
    .reset_index(name="n_countries")
)
print(eco_counts.to_string(index=False))

no_type = master[master["economy_type"].isna()]["iso3"].unique().tolist()
if no_type:
    print(f"\nCountries with no economy_type ({len(no_type)}): {sorted(no_type)}")
else:
    print("\nAll countries have an economy_type classification.")

In [ ]:
# ── 9.5  Duplicate (iso3, year) check ────────────────────────────────────────
dupes = master[master.duplicated(["iso3", "year"], keep=False)]

if dupes.empty:
    print("Duplicate check PASSED — no duplicate (iso3, year) pairs.")
else:
    print(f"DUPLICATE CHECK FAILED — {len(dupes)} rows with duplicate (iso3, year).")
    print("These must be resolved before training.")
    print(dupes[["iso3", "year"]].to_string(index=False))
    raise ValueError(
        f"{len(dupes)} duplicate (iso3, year) rows found in master panel. "
        "Investigate before proceeding."
    )

---
## Step 10 — Save

We save two files:

1. **`master_panel.csv`** — the full panel dataset, no index column.
2. **`master_panel_metadata.txt`** — a plain-text record of dimensions, year range,  
   column inventory grouped by source, and the consolidation logic applied in Step 8.  
   This becomes part of the dissertation methodology appendix.

In [ ]:
MASTER_FILE = PROC / "master_panel.csv"

master.to_csv(MASTER_FILE, index=False)
size_mb = MASTER_FILE.stat().st_size / (1024 ** 2)

print("=" * 60)
print("SAVED: master_panel.csv")
print("=" * 60)
print(f"  Path      : {MASTER_FILE}")
print(f"  Shape     : {master.shape[0]:,} rows × {master.shape[1]} columns")
print(f"  File size : {size_mb:.2f} MB")

In [ ]:
META_FILE = PROC / "master_panel_metadata.txt"

# Group columns by source for the metadata file
col_groups = {
    "Identifiers": ["iso3", "year"],
    "WEO (IMF World Economic Outlook)": [
        c for c in master.columns if c.startswith("weo_")
    ],
    "IMF Central Government Debt": ["imf_cg_debt_to_gdp"],
    "World Bank WDI": [
        c for c in master.columns if c.startswith("wdi_")
    ],
    "EM-DAT Climate Disasters": [
        c for c in master.columns if c.startswith("climate_")
    ],
    "ND-GAIN": [
        c for c in master.columns if c.startswith("ndgain_")
    ],
    "Income Classification (WB)": [
        "country_name", "Region", "Income group", "Lending category", "economy_type"
    ],
    "INFORM Risk Index": [
        c for c in master.columns if c.startswith("inform_")
    ],
    "Consolidated columns": [
        "debt_to_gdp", "gdp_growth", "inflation", "current_account_gdp"
    ],
}

meta_lines = [
    "MASTER PANEL METADATA",
    "=" * 60,
    f"Generated   : {date.today().isoformat()}",
    f"Rows        : {master.shape[0]:,}",
    f"Columns     : {master.shape[1]}",
    f"Countries   : {master['iso3'].nunique()}",
    f"Year range  : {int(master['year'].min())} – {int(master['year'].max())}",
    f"File size   : {size_mb:.2f} MB",
    "",
    "COLUMN INVENTORY BY SOURCE",
    "-" * 60,
]

for group, cols in col_groups.items():
    present = [c for c in cols if c in master.columns]
    meta_lines.append(f"\n[{group}]  ({len(present)} columns)")
    for c in present:
        pct_missing = master[c].isna().mean() * 100
        meta_lines.append(f"  {c:<45}  {pct_missing:5.1f}% missing")

meta_lines += [
    "",
    "CONSOLIDATED COLUMN PRIORITY LOGIC",
    "-" * 60,
    "debt_to_gdp:",
    "  1. imf_cg_debt_to_gdp  (IMF central govt debt — best cross-country comparability)",
    "  2. weo_govt_gross_debt  (WEO general govt gross debt — broader but well-documented)",
    "  3. wdi_debt_to_gdp      (World Bank WDI — most sparse, last resort)",
    "",
    "gdp_growth:",
    "  1. weo_gdp_constant pct_change()  (computed within-country, consistent deflation methodology)",
    "  2. wdi_gdp_growth                 (WDI direct — for countries absent from WEO)",
    "",
    "inflation:",
    "  1. weo_inflation_cpi   (WEO CPI average — widest country-year coverage)",
    "  2. wdi_inflation_cpi   (WDI CPI — fallback for non-WEO economies)",
    "",
    "current_account_gdp:",
    "  1. weo_current_account      (WEO — widest coverage)",
    "  2. wdi_current_account_gdp  (WDI — fallback)",
]

META_FILE.write_text("\n".join(meta_lines), encoding="utf-8")
print(f"Metadata saved: {META_FILE}")
print()
print("\n".join(meta_lines))